In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt
from opacus import PrivacyEngine
import optuna
import sys
from functools import partial
import random
import numpy as np
import time
import os
import pickle

# Experiment parameters

In [ ]:
def seed_everything(seed):
    # Set random seed for reproducibility
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

In [ ]:
os.makedirs('./toy-model-data', exist_ok=True)

seed = 42

seed_everything(seed)

# Parameters
d = 10  # Dimensionality
n = 1250  # Number of data points
epochs = 10
learning_rate = 1e-1
batch_size = 32

# Data generation

In [ ]:
# Generate synthetic data
mean = torch.zeros(d)  # Zero mean
cov = torch.eye(d)  # Standard Gaussian
data = torch.distributions.MultivariateNormal(mean, cov).sample((n,))
target = mean.repeat(n, 1)  # Our target is the true mean repeated

# Create DataLoader
dataset = TensorDataset(data, target)

# Split the dataset into training and validation
train_size = int(0.8 * n)  # 80% for training
val_size = n - train_size  # 20% for validation

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Display the sizes of each dataset for verification
print(f'Training dataset size: {len(train_dataset)}')
print(f'Validation dataset size: {len(val_dataset)}')

# Model

In [ ]:
# Define a simple linear model
class LinearModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Linear(input_dim, input_dim, bias=False)

        # Initialize weights to 1
        torch.nn.init.constant_(self.fc.weight, 1.0)

    def forward(self, x):
        return self.fc(x)

In [ ]:
def train_dp_model(train_loader, val_loader, epochs, learning_rate, max_grad_norm, epsilon):
    #print(f'Running with maximum gradient norm: {max_grad_norm}')

    # Initialize model and optimizer
    model = LinearModel(d)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Initialize Privacy Engine
    privacy_engine = PrivacyEngine()
    model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
        module=model,
        optimizer=optimizer,
        data_loader=train_loader,
        target_epsilon=epsilon,
        target_delta=1e-5,
        max_grad_norm=max_grad_norm,
        epochs=epochs,
        normalize_clipping=True,
    )

    criterion = nn.MSELoss()
    train_losses = []
    val_losses = []

    # Training and validation loop
    for epoch in range(epochs):        
        # Training phase
        model.train()  # Set model to training mode
        epoch_train_loss = 0
        for batch_data, batch_target in train_loader:
            optimizer.zero_grad()
            output = model(batch_data)
            loss = criterion(output, batch_target)
            loss.backward()
            optimizer.step()

            epoch_train_loss += loss.item()

        avg_train_loss = epoch_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation phase
        model.eval()  # Set model to evaluation mode
        epoch_val_loss = 0
        with torch.no_grad():  # Disable gradient calculation for validation
            for val_data, val_target in val_loader:
                val_output = model(val_data)
                val_loss = criterion(val_output, val_target)
                epoch_val_loss += val_loss.item()

        avg_val_loss = epoch_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        #print(f'Epoch [{epoch+1}/{epochs}], Training Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}')

    return train_losses[-1], val_losses[-1]  # Return the final losses

In [ ]:
def plot_results(max_grad_norms, results, epsilon):
    plt.figure(figsize=(10, 6))
    plt.plot(max_grad_norms, results, label=f"Epsilon = {epsilon}", marker='o')
    plt.xscale('log')
    plt.xlabel('Maximum Gradient Norm')
    plt.ylabel('MSE Loss')

    min_mse = min(results)
    plt.title(f'Effect of Maximum Gradient Norm on Toy model (epsilon={epsilon}) (Min MSE: {min_mse:.04})')
    plt.legend()
    plt.savefig(f'temp/toy-model-epsilon-{epsilon}.png')
    plt.grid(True)
    plt.show()

In [ ]:
def plot_results_combined(max_grad_norms, train_results_dict, val_results_dict):
    plt.figure(figsize=(12, 8))
    
    # Use a color map to ensure consistent colors for each epsilon
    colors = plt.cm.viridis(np.linspace(0, 1, len(train_results_dict)))

    for i, (epsilon, train_losses) in enumerate(train_results_dict.items()):
        val_losses = val_results_dict[epsilon]

        # Plot training losses with dotted lines
        plt.plot(max_grad_norms, train_losses, linestyle='--', color=colors[i], marker='o', label=f"Training Loss (ε = {epsilon})")
        
        # Plot validation losses with solid lines
        plt.plot(max_grad_norms, val_losses, linestyle='-', color=colors[i], marker='x', label=f"Validation Loss (ε = {epsilon})")

    plt.xscale('log')
    plt.xlabel('Maximum Gradient Norm')
    plt.ylabel('MSE Loss')
    plt.title('Effect of Maximum Gradient Norm on Toy Model')
    plt.legend()
    plt.grid(True)
    plt.ylim([0,5])
    plt.savefig(f'temp/toy-model-all-epsilons.png')
    plt.show()


In [ ]:
def experiment1():
    max_grad_norms = np.geomspace(1e-4, 30, 15)
    epsilon_values = [0.25, 1, 4, 8, 16]
    
    # Store results for all epsilon values
    train_results_dict = {}
    val_results_dict = {}

    for epsilon in epsilon_values:
        print(f'Running for epsilon: {epsilon}')

        seed_everything(seed)
        
        # Compute results for each max_grad_norm
        results = [
            train_dp_model(train_loader, val_loader, epochs=epochs, learning_rate=learning_rate, max_grad_norm=mgn, epsilon=epsilon)
            for mgn in max_grad_norms
        ]
        
        # Unpack results into separate lists for training and validation losses
        train_losses, val_losses = zip(*results)
        
        # Store the results in the dictionaries
        train_results_dict[epsilon] = train_losses
        val_results_dict[epsilon] = val_losses
    
    # Plot all results in a single plot
    plot_results_combined(max_grad_norms, train_results_dict, val_results_dict)

In [ ]:
def plot_mean_with_confidence_intervals(max_grad_norms, all_results_dict, n_repeats, save_fname=None):
    means = []
    conf_intervals = []

    # Calculate mean and confidence intervals for each max_grad_norm
    for mgn in max_grad_norms:
        losses = all_results_dict[mgn]
        mean_loss = np.mean(losses)
        std_dev = np.std(losses)
        conf_interval = 1.96 * (std_dev / np.sqrt(len(losses)))

        means.append(mean_loss)
        conf_intervals.append(conf_interval)

    # Plot mean line with confidence intervals
    plt.figure(figsize=(12, 8))
    plt.plot(max_grad_norms, means, color='b', marker='o', label='Mean MSE Loss')
    plt.fill_between(max_grad_norms, np.array(means) - np.array(conf_intervals), np.array(means) + np.array(conf_intervals), color='b', alpha=0.2, label='95% CI')
    
    plt.xscale('log')
    plt.xlabel('Maximum Gradient Norm')
    plt.ylabel('Mean MSE Loss')
    plt.title(f'Mean MSE Loss with 95% Confidence Intervals ({n_repeats} repeats) (ε = {epsilon})')
    plt.legend()
    plt.ylim([0,5])
    plt.grid(True)
    
    if save_fname:
        plt.savefig(save_fname)

    plt.show()


In [ ]:
def plot_all_repeats(max_grad_norms, all_results, n_repeats, save_fname=None):
    plt.figure(figsize=(12, 8))

    # Iterate over each repeat
    for i, repeat_losses in enumerate(all_results):
        # Plot each repeat's results as a separate line
        plt.plot(max_grad_norms, repeat_losses, alpha=0.5, color='blue', linewidth=0.5)

    plt.xscale('log')
    plt.xlabel('Maximum Gradient Norm')
    plt.ylabel('MSE Loss')
    plt.title(f'All {n_repeats} Repeats for MSE Losses (ε = {epsilon})')
    plt.grid(True)
    plt.ylim([0,5])
    
    if save_fname:
        plt.savefig(save_fname)

    plt.show()

In [ ]:
def experiment2():
    n_repeats = 100
    epsilon = 0.25
    max_grad_norms = np.geomspace(1e-4, 30, 15)
    all_results_dict = {mgn: [] for mgn in max_grad_norms}  # For confidence intervals
    all_results_list = []  # For plotting all the repeats separately
    
    for i in range(n_repeats):
        print(f'* Running repeat {i+1}..')
        repeat_losses = []
        for mgn in max_grad_norms:

            seed_everything(seed+i)

            # Replace the following line with your actual training function call
            train_loss, val_loss = train_dp_model(
                train_loader,
                val_loader,
                epochs=epochs,
                learning_rate=learning_rate,
                max_grad_norm=mgn,
                epsilon=epsilon,
            )
            all_results_dict[mgn].append(val_loss)  # Collect validation losses
            repeat_losses.append(val_loss)
            
        all_results_list.append(repeat_losses)
    
    plot_all_repeats(
        max_grad_norms,
        all_results_list,
        n_repeats,
        save_fname=f'temp/toy-model-epsilon-{epsilon}-{n_repeats}-repeats-raw-data.png',
    )

    plot_mean_with_confidence_intervals(
        max_grad_norms,
        all_results_dict,
        n_repeats,
        save_fname=f'temp/toy-model-epsilon-{epsilon}-{n_repeats}-repeats-mean-with-confidence.png',
    )


In [ ]:
def experiment3():
    initial_repeats = 100
    final_repeats = 800

    max_grad_norms = np.geomspace(1e-4, 30, 15)

    # Initialize result containers
    all_results_dict = {mgn: [] for mgn in max_grad_norms}  # For confidence intervals
    all_results_list = []  # For plotting all the repeats separately
    
    def run_experiments(n_repeats):        
        for i in range(n_repeats):
            print(f'* Running repeat {i+1}..')
            seed_everything(seed+i)

            repeat_losses = []
            for mgn in max_grad_norms:
                # Replace the following line with your actual training function call
                train_loss, val_loss = train_dp_model(
                    train_loader,
                    val_loader,
                    epochs=epochs,
                    learning_rate=learning_rate,
                    max_grad_norm=mgn,
                    epsilon=epsilon,
                )
                
                print(f'  - mgn: {mgn}, val_loss: {val_loss}')  # Debug line

                all_results_dict[mgn].append(val_loss)  # Collect validation losses
                repeat_losses.append(val_loss)
                
            print(f'Repeat {i+1} losses: {repeat_losses}')  # Debug line
            all_results_list.append(repeat_losses)
    
    current_repeats = initial_repeats
    while current_repeats <= final_repeats:
        print(f'Running with {current_repeats} repeats...')
        run_experiments(current_repeats)
        
        print(f'All results list after {current_repeats} repeats: {all_results_list}')  # Debug line
        print(f'All results dict after {current_repeats} repeats: {all_results_dict}')  # Debug line
        
        plot_all_repeats(
            max_grad_norms,
            all_results_list,
            current_repeats,
            f'temp/toy-model-epsilon-{epsilon}-{current_repeats}-repeats-raw-data.png',
        )
        plot_mean_with_confidence_intervals(
            max_grad_norms,
            all_results_dict,
            current_repeats,
            save_fname=f'temp/toy-model-epsilon-{epsilon}-{current_repeats}-repeats-mean-with-confidence.png',
        )
    
        current_repeats *= 2


In [ ]:
def objective(trial: optuna.trial.Trial, max_grad_norm: float):
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1.0)
    batch_size = trial.suggest_int('batch_size', 8, 1000)
    epochs = 10
    epsilon = 0.25
    n = 1000
    d = 10

    print(f'*** Running with max_grad_norm: {max_grad_norm}, learning_rate: {learning_rate}, batch_size: {batch_size}')
    train_loss, val_loss = train_dp_model(train_loader, val_loader, epochs, learning_rate, max_grad_norm, epsilon)
    print(f'****** Training Loss: {train_loss}, Validation Loss: {val_loss}')

    return val_loss  # We want to minimize validation loss

In [ ]:
def optimize(max_grad_norm, n_trials: int = 20, direction: str ='minimize', seed: int = 42):
    seed_everything(seed)

    sampler_cls = getattr(optuna.integration, 'BoTorchSampler')
    sampler = sampler_cls(seed=seed)

    study = optuna.create_study(
        study_name='dp_toy_model_optimization',
        sampler=sampler,
        direction=direction,
    )

    objective_func = partial(objective, max_grad_norm=max_grad_norm)
    
    study.optimize(
        objective_func,
        n_trials=n_trials,
        gc_after_trial=True,  # garbage collect after each trial
    )

    trial = study.best_trial
    print(f'Best objective value: {trial.value}', file=sys.stderr)
    print('Best parameters:', file=sys.stderr)
    for key, value in trial.params.items():
        print(f' - {key}: {value}', file=sys.stderr)

    return trial.value, trial.params

In [ ]:
def plot_losses_vs_max_grad_norms(max_grad_norm_values, optimized_losses):
    plt.figure(figsize=(10, 6))
    plt.plot(max_grad_norm_values, optimized_losses, marker='o', linestyle='-', color='b')
    plt.xscale('log')
    plt.xlabel('Max Gradient Norm')
    plt.ylabel('Loss')
    plt.title('Loss after HPO as a Function of Max Gradient Norm')
    plt.grid(True)
    #plt.ylim([0, 5])
    plt.savefig('temp/toy-model-epsilon-0.25-HPO.png')
    plt.show()


In [ ]:
def experiment4():
    max_grad_norm_values = np.geomspace(1e-4, 30, 15)
    optimized_losses = []
    optimized_params = []
    
    total_start_time = time.time()
    
    for max_grad_norm in max_grad_norm_values:
        print(f'Optimizing for max_grad_norm: {max_grad_norm}')
        
        study_start_time = time.time()
        best_loss, best_params = optimize(max_grad_norm, n_trials=1, seed=seed)
        study_duration = time.time() - study_start_time
        print(f'Study for max_grad_norm {max_grad_norm} took {study_duration:.2f} seconds')
        
        optimized_losses.append(best_loss)
        optimized_params.append(best_params)
    
    total_duration = time.time() - total_start_time
    print(f'Total runtime for all studies: {total_duration:.2f} seconds')

    with open('./toy-model-data/optimized_losses.pkl', 'wb') as f:
        pickle.dump(optimized_losses, f)
    
    # Save optimized_params
    with open('./toy-model-data/optimized_params.pkl', 'wb') as f:
        pickle.dump(optimized_params, f)
    
    plot_losses_vs_max_grad_norms(max_grad_norm_values, optimized_losses)

Max Grad Norm: 0.00367190796352817
- Default hypers: 1.4226503372192383
- 20 trials: 1.4968048334121704
- 50 trials: 0.8834649324417114
- 100 trials: 0.6916019320487976
- 200 trials: 0.6680276989936829

In [ ]:
def experiment_with_optimized_hypers(max_grad_norm_values, optimized_params, n, d, epochs, epsilon):
    initial_repeats = 100
    final_repeats = 1600
        
    # Store results here
    all_results_dict = {mgn: [] for mgn in max_grad_norm_values}  # For confidence intervals
    all_results_list = []  # For plotting all the repeats separately
    
    # Function to incrementally run experiments
    def run_experiments(n_repeats):
        for i in range(n_repeats):
            print(f'* Running repeat {i+1}..')
            seed_everything(seed+i)
            repeat_losses = []
            for idx, mgn in enumerate(max_grad_norm_values):
                # Get optimized hyperparameters for this max_grad_norm
                params = optimized_params[idx]
                learning_rate = params['learning_rate']
                batch_size = params['batch_size']

                # Create DataLoaders
                train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
                val_loader = DataLoader(val_dataset, batch_size=250, shuffle=False)

                # Train the model
                train_loss, val_loss = train_dp_model(train_loader, val_loader, epochs, learning_rate, mgn, epsilon)
                
                all_results_dict[mgn].append(val_loss)  # Collect validation losses
                repeat_losses.append(val_loss)
                
            all_results_list.append(repeat_losses)
    
    # Main loop to double the repeats
    current_repeats = initial_repeats
    while current_repeats <= final_repeats:
        print(f'Running with {current_repeats} repeats...')
        run_experiments(current_repeats)
    
        # Plot the results after each iteration
        plot_all_repeats(
            max_grad_norm_values,
            all_results_list,
            current_repeats,
            save_fname=f'temp/toy-model-HPO-epsilon-{epsilon}-{current_repeats}-repeats-raw-data.png',
        )
        plot_mean_with_confidence_intervals(
            max_grad_norm_values,
            all_results_dict,
            current_repeats,
            save_fname=f'temp/toy-model-HPO-epsilon-{epsilon}-{current_repeats}-repeats-mean-with-confidence.png'
        )
    
        # Double the number of repeats for the next iteration
        current_repeats *= 2
    
    return all_results_list, all_results_dict

In [ ]:
with open('./toy-model-data/optimized_params.pkl', 'rb') as f:
    optimized_params = pickle.load(f)

with open('./toy-model-data/optimized_losses.pkl', 'rb') as f:
    optimized_losses = pickle.load(f)

In [ ]:
max_grad_norm_values = np.geomspace(1e-4, 30, 15)
epsilon = 0.25

all_results_list, all_results_dict = experiment_with_optimized_hypers(
    max_grad_norm_values,
    optimized_params,
    n,
    d,
    epochs,
    epsilon,
)

In [ ]:
with open('./toy-model-data/optimized_losses.pkl', 'rb') as f:
    optimized_losses = pickle.load(f)

plot_losses_vs_max_grad_norms(max_grad_norm_values, optimized_losses)